# spancy_shift_spatial.py — equivalence check and training time

`spancy_shift_spatial.py` is the clean final module of the region GNN: same model, losses, sampler
and random-number streams as `spancy_shift_region.py` (`sampler='patches'`, `graph_mode='fixed'`),
with engineering changes only (Stage 1 + graph computed once, data on the GPU, reused index buffers,
correction computed once and scaled by α).

GPU message passing is not bit-reproducible run to run, so a full 10-epoch comparison cannot be exact.
Instead:
1. **§2 short equivalence:** both modules train 1 epoch × 5 steps from the same seed → losses and
   weights must agree to ~1e-4; both inference paths on the SAME weights must agree to ~1e-4 (log1p).
2. **§3 full training** of the new module at seed 6 → training time for the article, correction size
   (region_final seed 6: uni_delta 0.0412).

Upload `spancy_shift.py`, `spancy_shift_region.py` (with `NeighbourPatchSampler`) and
`spancy_shift_spatial.py`. ~25 min. **Paste back §2 and §3.**

## 0. Colab Setup

In [ ]:
# Install torch-geometric (auto-detect Colab's torch + CUDA version)
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q "pandas<3" "anndata<0.11" scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
from google.colab import files
import os

NEEDED = ['spancy_shift.py', 'spancy_shift_region.py', 'spancy_shift_spatial.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print(f'Please upload: {missing}')
    files.upload()
for f in NEEDED:
    assert os.path.exists(f), f'{f} not found — upload it (Colab renames re-uploads to "name (1).py")'
    print(f'{f:24s} {os.path.getsize(f):>10,} bytes')
assert 'class NeighbourPatchSampler' in open('spancy_shift_region.py').read(), 'OLD spancy_shift_region.py'
print('Modules: OK')

In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

import spancy_shift_region as sr      # reference implementation
import spancy_shift_spatial as spx    # clean final module
from spancy_shift_spatial import load_adata, detect_bimodal_markers, find_best_sample_per_marker

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load data, bimodal markers, reference samples

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

print(f"Bimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")
print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")
print("\nNote: DDPM scores the full joint distribution — no special handling required.")
bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]

In [ ]:
ref_sample_per_marker = find_best_sample_per_marker(adata)
uni_markers = [m for m in marker_names if m not in bimodal_names]
uni_idx = [marker_names.index(m) for m in uni_markers]
bim_idx = [marker_names.index(m) for m in bimodal_names]
print(f'{len(uni_markers)} unimodal markers; bimodal: {bimodal_names}')

## 2. Short equivalence check (same seed, 1 epoch × 5 steps)

In [ ]:
CFG = dict(n_epochs=1, lr=1e-3, warmup_epochs=1, n_per_batch=512, temperature=0.07, w_recon=0.1,
           w_contrast=0.5, w_adv=0.3, w_mmd=1.0, mmd_samples=256, grl_max=1.0)
N_STEPS = 5

prep = spx.prepare(adata, ref_sample_per_marker, k_neighbors=15, bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC)
print(f'prepare(): {prep.seconds:.0f} s (Stage 1 + scaling + region graph, once)')

torch.manual_seed(6)
torch.cuda.manual_seed_all(6)
m_old, sc_old, ref_old, h_old = sr.train(
    adata, device_str=DEVICE, bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
    ref_sample_per_marker=ref_sample_per_marker, k_neighbors=15, hybrid_alpha=0.6, n_patches=4,
    graph_mode='fixed', sampler='patches', max_steps_per_epoch=N_STEPS, **CFG)
torch.manual_seed(6)
torch.cuda.manual_seed_all(6)
m_new, h_new = spx.train(prep, device_str=DEVICE, max_steps_per_epoch=N_STEPS, **CFG)

print('\nlosses (old vs new):')
for k in ['loss', 'recon', 'contrast', 'adv', 'mmd']:
    print(f'  {k:9s} {h_old[k][0]:.6f}  {h_new[k][0]:.6f}  diff {abs(h_old[k][0] - h_new[k][0]):.2e}')
p_old = dict(m_old.named_parameters())
w_diff = max(float((p_old[n].detach().cpu() - p.detach().cpu()).abs().max()) for n, p in m_new.named_parameters())
print(f'max |weight difference| after {N_STEPS} steps: {w_diff:.2e}')
print('Stage 1 identical:', np.array_equal(
    np.asarray(prep.adata_base.layers['normalized_base']),
    np.asarray(sr.shift_normalize_per_marker(adata, ref_sample_per_marker,
                                             bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
                                             layer_name='normalized_base')[0].layers['normalized_base'])))

# inference: both paths on the SAME weights (the new model)
torch.manual_seed(100)
out_old = sr.normalize_adata(adata, m_new, prep.scaler, ref_sample_per_marker, hybrid_alpha=0.6, k_neighbors=15,
                             inference_batch_size=2048, device_str=DEVICE, layer_name='normalized',
                             keep_base_layer=True, bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
                             graph_mode='fixed')
t1 = time.time()
delta = spx.correct(prep, m_new, device_str=DEVICE)
X_new = spx.apply(prep, delta, alpha=0.6)
t_new = time.time() - t1
L_old = np.log1p(np.asarray(out_old.layers['normalized'], dtype=np.float64))
L_new = np.log1p(X_new.astype(np.float64))
print(f'inference max |log1p difference| old vs new path (same weights): {np.abs(L_old - L_new).max():.2e}  '
      f'(new path {t_new:.0f} s incl. one network pass)')
loss_ok = all(abs(h_old[k][0] - h_new[k][0]) < 1e-3 for k in ['loss', 'mmd', 'contrast'])
print(f'\nEQUIVALENT: losses {loss_ok}, weights {w_diff < 1e-3}, inference {np.abs(L_old - L_new).max() < 1e-3}')
del m_old, sc_old, ref_old, h_old, out_old, L_old, L_new, X_new, delta
gc.collect()

## 3. Full training with the new module (seed 6, 10 epochs) — time for the article

In [ ]:
torch.manual_seed(6)
torch.cuda.manual_seed_all(6)
model, hist = spx.train(prep, device_str=DEVICE, n_epochs=10, lr=1e-3, warmup_epochs=2, n_per_batch=512,
                        temperature=0.07, w_recon=0.1, w_contrast=0.5, w_adv=0.3, w_mmd=1.0,
                        mmd_samples=256, grl_max=1.0)
t1 = time.time()
delta = spx.correct(prep, model, device_str=DEVICE)
t_corr = time.time() - t1
X06 = spx.apply(prep, delta, alpha=0.6)
uni = ~prep.is_bimodal
base_log = np.log1p(np.clip(np.asarray(prep.adata_base.layers['normalized_base']), 0, None))
uni_delta = float(np.abs(np.log1p(X06[:, uni]) - base_log[:, uni]).mean())
bim_delta = float(np.abs(np.log1p(X06[:, ~uni]) - base_log[:, ~uni]).mean()) if (~uni).any() else 0.0
print(f'prepare {prep.seconds / 60:.1f} min (once) | training {hist["train_min"][0]:.1f} min '
      f'(old module: ~15 min) | correction {t_corr:.0f} s | any alpha afterwards: < 1 s')
print(f'uni_delta {uni_delta:.4f} (region_final s6: 0.0412)   bim_delta {bim_delta:.4f} (must be 0)')
print('final epoch losses:', {k: round(v[-1], 4) for k, v in hist.items() if k != 'train_min'})